# THINGS Odd-One-Out: Human–Model Alignment — Colab Runner

Runs the full pipeline on a Colab GPU. The only manual step is the THINGS
**images** (license click-through). Everything else — behavioral data, training,
transfer, analysis — is scripted. The smoke-test section validates the whole
Phase 3–5 pipeline **without images** using synthetic features.

**Tip:** Runtime → Change runtime type → GPU (T4).

## 1. Clone + install

In [ ]:
!git clone https://github.com/Sudarssan-N/VisionModal-HumalObjectSimilarity.git
%cd VisionModal-HumalObjectSimilarity
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Download + prepare behavioral data (scripted, ~412 MB)

In [ ]:
!python src/download_data.py
!cd data/raw && unzip -q -o osfstorage-archive.zip -d extracted
!cd data/raw/extracted && unzip -q -o full_triplet_dataset.zip -d triplets
!python src/prepare_data.py

## 3. Smoke test — validate Phases 3–5 WITHOUT images

Fabricates features that are a noisy linear function of the real human
embedding, then runs the real training/transfer/analysis scripts. Confirms the
pipeline works on this machine before investing in feature extraction.

In [ ]:
!python src/smoke_test.py --force

## 4. (Real run) Images → features

Download the THINGS / THINGSplus images (see `DATA_SETUP.md`) to this Colab
session — e.g. upload a zip to `/content` or mount Google Drive — then point
`organize_images.py` at the `object_images` folder.

In [ ]:
# Example (adjust the path to wherever you placed the images):
# from google.colab import drive; drive.mount('/content/drive')
# !python src/organize_images.py /content/drive/MyDrive/THINGS/object_images
# !python src/data.py

In [ ]:
# Phase 1 + 2: extract embeddings, zero-shot baseline
# !python src/extract_features.py
# !python src/zeroshot_eval.py

In [ ]:
# Phase 3–5: alignment, transfer, analysis
# !python src/train_transform.py
# !python src/transfer.py
# !python src/analysis.py

In [ ]:
# View results
import json, glob
for f in sorted(glob.glob('results/*.json')):
    print('==', f); print(json.dumps(json.load(open(f)), indent=2)[:1500])